# MLA Phase 5 — XLM-R (Train/Dev only)

Thí nghiệm tái lập `FacebookAI/xlm-roberta-base` trên split đã kiểm toán.

**Bảo vệ Test:** Notebook chỉ nạp `train_segments.csv` và `dev_segments.csv`. Không tải, đọc, dự đoán hoặc đánh giá Test.

Cấu hình: max_length=256; train batch=8; eval batch=16; gradient accumulation=2; 3 epochs; learning rate=2e-5; weight decay=0.01; seed=42; weighted cross-entropy. Chọn checkpoint theo Dev Average Precision (`eval_pr_auc`).

## 0. Chuẩn bị Colab

Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU miễn phí khả dụng). Trong Google Drive tạo `MyDrive/MLA_Phase5_XLMR/data/` và đặt vào đó đúng hai file `train_segments.csv`, `dev_segments.csv`. Không đưa Test vào thư mục.

In [10]:
!pip -q install "transformers==4.46.3" "accelerate==1.1.1" "scikit-learn>=1.5,<2" "sentencepiece>=0.2"

In [11]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/MLA_Phase5_XLMR")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / "train_segments.csv"
DEV_PATH = DATA_DIR / "dev_segments.csv"
assert TRAIN_PATH.is_file(), f"Missing {TRAIN_PATH}"
assert DEV_PATH.is_file(), f"Missing {DEV_PATH}"
print("Train:", TRAIN_PATH)
print("Dev:", DEV_PATH)
print("Output:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: /content/drive/MyDrive/MLA_Phase5_XLMR/data/train_segments.csv
Dev: /content/drive/MyDrive/MLA_Phase5_XLMR/data/dev_segments.csv
Output: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs


In [12]:
import os, json, random, platform, time, math
import numpy as np
import pandas as pd
import torch, transformers, accelerate, sklearn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) if torch.cuda.is_available() else None

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime > Change runtime type > GPU rồi chạy lại.")
!nvidia-smi

Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.46.3
Accelerate: 1.1.1
CUDA: True
Fri Nov 20 12:29:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             28W /   70W |    6505MiB /  15360MiB |      0%      Default |
|                                         |         

## 1. Nạp và xác minh dữ liệu Train/Dev

In [13]:
REQUIRED = {"system","doc","doc_id","seg_id","line","source","target","distinct_raters","serious_votes","y_primary"}
train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)

for name, df in [("Train", train_df), ("Dev", dev_df)]:
    assert REQUIRED.issubset(df.columns), f"{name}: thiếu cột {REQUIRED-set(df.columns)}"
    assert df["y_primary"].notna().all()
    assert set(df["y_primary"].unique()).issubset({0,1})
    assert df[["source","target"]].notna().all().all()
    assert (df["source"].astype(str).str.strip() != "").all()
    assert (df["target"].astype(str).str.strip() != "").all()

assert len(train_df)==6797 and int(train_df.y_primary.sum())==803 and train_df.doc.nunique()==90
assert len(dev_df)==1554 and int(dev_df.y_primary.sum())==218 and dev_df.doc.nunique()==20
assert set(train_df.doc).isdisjoint(set(dev_df.doc)), "Document overlap!"

print("Train:", train_df.shape, "docs:", train_df.doc.nunique(), "positives:", int(train_df.y_primary.sum()), "rate:", train_df.y_primary.mean())
print("Dev:", dev_df.shape, "docs:", dev_df.doc.nunique(), "positives:", int(dev_df.y_primary.sum()), "rate:", dev_df.y_primary.mean())
print("Systems:", train_df.system.nunique(), dev_df.system.nunique())
print("PASS: counts and document split match audited data.")

Train: (6797, 10) docs: 90 positives: 803 rate: 0.11814035603942916
Dev: (1554, 10) docs: 20 positives: 218 rate: 0.1402831402831403
Systems: 7 7
PASS: counts and document split match audited data.


## 2. Tokenize cặp Source–MT Target (max_length=256)

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
from torch.utils.data import Dataset

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 256
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    def __init__(self, frame):
        self.labels = frame.y_primary.astype(int).tolist()
        self.enc = tokenizer(
            frame.source.astype(str).tolist(),
            frame.target.astype(str).tolist(),
            truncation=True, max_length=MAX_LENGTH, padding=False
        )
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        x = {k: v[i] for k,v in self.enc.items()}
        x["labels"] = self.labels[i]
        return x

train_ds = PairDataset(train_df)
dev_ds = PairDataset(dev_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized:", len(train_ds), len(dev_ds))

Tokenized: 6797 1554


## 3. Mô hình và weighted cross-entropy (class weights chỉ tính từ Train)

In [15]:
weights_np = compute_class_weight(
    class_weight="balanced", classes=np.array([0,1]),
    y=train_df.y_primary.astype(int).to_numpy()
)
class_weights = torch.tensor(weights_np, dtype=torch.float)
print("Class weights [0,1]:", weights_np.tolist())

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:,1]
    pred = (probs >= 0.5).astype(int)
    return {
        "pr_auc": float(average_precision_score(labels, probs)),
        "roc_auc": float(roc_auc_score(labels, probs)),
        "precision_at_05": float(precision_score(labels,pred,zero_division=0)),
        "recall_at_05": float(recall_score(labels,pred,zero_division=0)),
        "mcc_at_05": float(matthews_corrcoef(labels,pred))
    }

Class weights [0,1]: [0.5669836503169836, 4.232254047322541]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Huấn luyện 3 epoch; chọn checkpoint bằng Dev PR-AUC

In [16]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_pr_auc",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    optim="adamw_torch"
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics, class_weights=class_weights
)
start = time.time()
train_result = trainer.train()
training_seconds = time.time() - start
print("Training minutes:", round(training_seconds/60,2))
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best Dev AP:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Pr Auc,Roc Auc,Precision At 05,Recall At 05,Mcc At 05
1,0.670300,0.757186,0.175143,0.542218,0.200000,0.013761,0.016976
2,0.567300,0.874378,0.201489,0.645807,0.258503,0.174312,0.110033
3,0.440100,1.279431,0.179628,0.553897,0.254717,0.123853,0.089154


Training minutes: 8.63
Best checkpoint: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/checkpoints/checkpoint-850
Best Dev AP: 0.20148948795101473


## 5. Đánh giá ranking trên Dev

In [17]:
pred_out = trainer.predict(dev_ds)
dev_probs = torch.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()[:,1]
y = dev_df.y_primary.astype(int).to_numpy()

def top20_metrics(y_true, scores):
    n = len(y_true); k = int(np.ceil(0.20*n))
    order = np.argsort(-scores, kind="mergesort")
    chosen = order[:k]
    selected = y_true[chosen]
    positives = int(y_true.sum())
    tp = int(selected.sum())
    precision = tp/k
    recall = tp/positives if positives else 0.0
    prevalence = positives/n
    lift = precision/prevalence if prevalence else 0.0
    selected_mask = np.zeros(n, dtype=int); selected_mask[chosen] = 1
    return {
        "k_top20": k, "true_positives_top20": tp,
        "precision_at_20": precision, "recall_at_20": recall,
        "lift_at_20": lift,
        "mcc_top20": float(matthews_corrcoef(selected_mask,y))
    }

metrics = {
    "n_dev": int(len(y)), "positives": int(y.sum()),
    "prevalence": float(y.mean()),
    "average_precision_pr_auc": float(average_precision_score(y,dev_probs)),
    "roc_auc": float(roc_auc_score(y,dev_probs)),
    **top20_metrics(y,dev_probs)
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_dev": 1554,
  "positives": 218,
  "prevalence": 0.1402831402831403,
  "average_precision_pr_auc": 0.20148948795101473,
  "roc_auc": 0.6458070098335439,
  "k_top20": 311,
  "true_positives_top20": 68,
  "precision_at_20": 0.21864951768488747,
  "recall_at_20": 0.3119266055045872,
  "lift_at_20": 1.5586300480840143,
  "mcc_top20": 0.11287400634091907
}


## 6. Lưu checkpoint tốt nhất, dự đoán Dev và log

In [18]:
BEST_DIR = OUTPUT_DIR / "best_model"
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))

dev_pred = dev_df[["system","doc","doc_id","seg_id","source","target","y_primary"]].copy()
dev_pred["score_XLMR"] = dev_probs
dev_pred["prediction_at_0_5"] = (dev_probs >= 0.5).astype(int)
dev_pred.to_csv(OUTPUT_DIR / "dev_predictions_xlmr.csv", index=False)

with open(OUTPUT_DIR / "dev_metrics_xlmr.json","w",encoding="utf-8") as f:
    json.dump(metrics,f,indent=2,ensure_ascii=False)

summary = {
    "model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
    "train_batch_size": 8, "eval_batch_size": 16,
    "gradient_accumulation_steps": 2, "epochs": 3,
    "learning_rate": 2e-5, "weight_decay": 0.01,
    "class_weights": weights_np.tolist(),
    "training_seconds": training_seconds,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
    "train_rows": len(train_df), "dev_rows": len(dev_df),
    "test_accessed": False
}
with open(OUTPUT_DIR / "training_summary.json","w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)
pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR/"training_log_history.csv",index=False)

assert len(dev_pred)==1554 and dev_pred.score_XLMR.notna().all()
assert not (OUTPUT_DIR/"test_predictions_xlmr.csv").exists()
print("PASS: Saved 1,554 Dev predictions. Test was not accessed.")
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

PASS: Saved 1,554 Dev predictions. Test was not accessed.
Files: ['checkpoints', 'best_model', 'dev_predictions_xlmr.csv', 'dev_metrics_xlmr.json', 'training_summary.json', 'training_log_history.csv']


In [19]:
import pandas as pd

log_df = pd.DataFrame(trainer.state.log_history)

log_path = "/content/drive/MyDrive/MLA_Phase5_XLMR/outputs/training_log_history.csv"
log_df.to_csv(log_path, index=False)

print(log_df.to_string(index=False))
print("Đã lưu:", log_path)

  loss  grad_norm  learning_rate    epoch  step  eval_loss  eval_pr_auc  eval_roc_auc  eval_precision_at_05  eval_recall_at_05  eval_mcc_at_05  eval_runtime  eval_samples_per_second  eval_steps_per_second  train_runtime  train_samples_per_second  train_steps_per_second   total_flos  train_loss
0.6902  17.946112   1.926275e-05 0.117647    50        NaN          NaN           NaN                   NaN                NaN             NaN           NaN                      NaN                    NaN            NaN                       NaN                     NaN          NaN         NaN
0.7091   4.374155   1.849412e-05 0.235294   100        NaN          NaN           NaN                   NaN                NaN             NaN           NaN                      NaN                    NaN            NaN                       NaN                     NaN          NaN         NaN
0.6645   5.999828   1.770980e-05 0.352941   150        NaN          NaN           NaN                   NaN        

## 7. Kết quả cần tải về để kiểm toán Phase 5

Trong Google Drive, thư mục `MLA_Phase5_XLMR/outputs/` sẽ có:
- `best_model/` — checkpoint tốt nhất theo Dev AP.
- `dev_predictions_xlmr.csv` — dự đoán 1.554 mẫu Dev.
- `dev_metrics_xlmr.json` — AP, ROC-AUC, Precision/Recall/Lift/MCC@20%.
- `training_summary.json` — cấu hình, class weights, thời gian huấn luyện.
- `training_log_history.csv` — lịch sử loss và metric.

**Diễn giải:** AP được tính bằng `average_precision_score` (Average Precision). Các metric này là kết quả Dev, không phải Test. Không dùng Dev để thay đổi nhãn hoặc split. Chỉ mở Test sau khi quy trình so sánh đã được phê duyệt.